A notebook for batch processing the ExoMars SKP Field Test 1 dataset into RGB products, using the fully automated '99u' (99th percentile unbalanced) stretch routine.

In [75]:
%load_ext autoreload
%autoreload 2
%config InlineBackend.figure_format = 'svg' # note - requires Inkscape to be installed, and to be accessible from the terminal
# on macos, add inkscape to the path with `sudo ln -s /Applications/Inkscape.app/Contents/MacOS/inkscape /usr/local/bin/inkscape`
%matplotlib inline

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [76]:
import aupy

# Setting Paths and Arranging the Data

Data from the ROCC comes in various directory structures, and with the filenames relabelled.

We need to reorganise and relabel this data, which can all be done via accessing the xml files attached with each image.

In steps:

1. Access directory for all data from the given sol.
2. Find all image data (as png's) within the sol directory.
3. Find the paired xml files for each image.
4. Extract filenames and other organisational metadata from the images and xml files.
5. Put the data into a new processing directory for that sol.

It seems that only WAC's ad some HRC have the metadata we are looking for, so we're going to only get images with HRC or WAC in the name (i.e. ignoring Nav and Loc Cams).

In [77]:
from pathlib import Path

In [78]:
data_dir = Path('/Users/sci/Documents/Documents - Roger’s MacBook Air/projects/esaskp_2025-2027/field_trials/SKP_FT1/data/downlink')

In [79]:
# make a list of all the directories in the data directory
sols = [d.name for d in data_dir.iterdir()]
# drop .DS_Store
sols = [s for s in sols if s != '.DS_Store']
sols.sort()

In [80]:
sols

['sol03',
 'sol04',
 'sol05',
 'sol06',
 'sol07',
 'sol38',
 'sol39',
 'sol40',
 'sol41']

Choose a Sol to process.

Let's develop on Sol41, because we have a range of cameras and scenes on this sol.

In [81]:
sol = sols[-3]
sol

'sol39'

In [82]:
sol_dir = data_dir / sol

The downlink will be arranged in various directories that might not find a pattern.

A clean way to deal with this is bypass the folders, and just search for the png.s

In [89]:
sol_dir

PosixPath('/Users/sci/Documents/Documents - Roger’s MacBook Air/projects/esaskp_2025-2027/field_trials/SKP_FT1/data/downlink/sol39')

In [84]:
pr_dir = sol_dir.parent.parent / 'processing'

In [90]:
sol_parser = aupy.ROCCDownlinkDirTool(sol_dir, pr_dir)
sol_parser.organise()

/opt/homebrew/Caskroom/miniconda/base/envs/aupy_dev/lib/python3.13/site-packages/IPython/extensions/deduperreload/deduperreload.py:290: DeprecationWarning: ast.Ellipsis is deprecated and will be removed in Python 3.14; use ast.Constant instead
  elif not isinstance(ast_elt, (ast.Ellipsis, ast.Pass)):
/opt/homebrew/Caskroom/miniconda/base/envs/aupy_dev/lib/python3.13/site-packages/IPython/extensions/deduperreload/deduperreload.py:290: DeprecationWarning: ast.Ellipsis is deprecated and will be removed in Python 3.14; use ast.Constant instead
  elif not isinstance(ast_elt, (ast.Ellipsis, ast.Pass)):
/opt/homebrew/Caskroom/miniconda/base/envs/aupy_dev/lib/python3.13/site-packages/IPython/extensions/deduperreload/deduperreload.py:290: DeprecationWarning: ast.Ellipsis is deprecated and will be removed in Python 3.14; use ast.Constant instead
  elif not isinstance(ast_elt, (ast.Ellipsis, ast.Pass)):
/opt/homebrew/Caskroom/miniconda/base/envs/aupy_dev/lib/python3.13/site-packages/IPython/exten

[(PosixPath('/Users/sci/Documents/Documents - Roger’s MacBook Air/projects/esaskp_2025-2027/field_trials/SKP_FT1/data/processing/sol39/aupy/sol39_00/sol39_00_00_T+05.04_P+040.47/sol00039_00000_00000_00001_LWAC4_T5.039999961853027_P40.470001220703125.png'),
  PosixPath('/Users/sci/Documents/Documents - Roger’s MacBook Air/projects/esaskp_2025-2027/field_trials/SKP_FT1/data/processing/sol39/aupy/sol39_00/sol39_00_00_T+05.04_P+040.47/sol00039_00000_00000_00001_LWAC4_T5.039999961853027_P40.470001220703125.png.xml')),
 (PosixPath('/Users/sci/Documents/Documents - Roger’s MacBook Air/projects/esaskp_2025-2027/field_trials/SKP_FT1/data/processing/sol39/aupy/sol39_00/sol39_00_00_T+05.04_P+040.47/sol00039_00000_00000_00002_LWAC5_T5.039999961853027_P40.470001220703125.png'),
  PosixPath('/Users/sci/Documents/Documents - Roger’s MacBook Air/projects/esaskp_2025-2027/field_trials/SKP_FT1/data/processing/sol39/aupy/sol39_00/sol39_00_00_T+05.04_P+040.47/sol00039_00000_00000_00002_LWAC5_T5.0399999618

Choose a scene to process.

In [87]:
scene = scenes[-2]
scene

NameError: name 'scenes' is not defined

Check if the scene contains any trial directories

In [ ]:
# look for subdirectorys in the scene directory
scene_dir = sol_dir / scene
# check that the scene directory exists
if not scene_dir.exists():
    raise FileNotFoundError(f"Scene directory {scene_dir} does not exist.")
# get the subdirectories in the scene directory
trials = [d.name for d in scene_dir.iterdir() if d.is_dir()]
trials.sort()

if trials == []:
    trials = ['']

trials

The batch script

In [ ]:
# for sol in sols:
#     # get the scenes in the sol directory
#     sol_dir = data_dir / sol
#     # check that the sol directory exists
#     if not sol_dir.exists():
#         raise FileNotFoundError(f"Sol directory {sol_dir} does not exist.")
#     # get the scenes in the sol directory
#     scenes = [d.name for d in sol_dir.iterdir() if d.is_dir()]
#     scenes.sort()
    
#     print(f'{sol}: {scenes}')
#     for scene in scenes:

# look for subdirectorys in the scene directory
scene_dir = sol_dir / scene
# check that the scene directory exists
if not scene_dir.exists():
    raise FileNotFoundError(f"Scene directory {scene_dir} does not exist.")
# get the subdirectories in the scene directory
trials = [d.name for d in scene_dir.iterdir() if d.is_dir()]
trials.sort()

if trials == []:
    trials = [None]

print(f'  {sol} {scene}: {trials}')        
for trial in trials:
    print(f'    {sol} {scene} {trial}')

    # # process HRC
    # hrc_loader = aupy.AupeIO('HRC', 'RGB',scene_dir, sol, scene, trial=trial)
    # if hrc_loader.input_files != []:            
    #     hrc_rgb = hrc_loader.load_frame()
    #     hrc_rgb.debayer()
    #     hrc_rgb.export_image('99s')
    #     hrc_rgb.export_image('bps')

    # process LWAC
    lwac_loader = aupy.AupeIO('LWAC', 'RGB',scene_dir, sol, scene, trial=trial)
    if lwac_loader.input_files != []:
        lwac_rgb = lwac_loader.load_frame()
        lwac_rgb.export_image('bpu', gamma=2.2)
        # lwac_rgb.export_image('bpb')

    # # process RWAC
    # rwac_loader = aupy.AupeIO('RWAC', 'RGB',scene_dir, sol, scene, trial=trial)
    # if rwac_loader.input_files != []:
    #     rwac_rgb = rwac_loader.load_frame()            
    #     rwac_rgb.export_image('99u')
    #     rwac_rgb.export_image('bpb')
